# Preço / Lucro Histórico de Empresas Listadas na B3 com Dados da CVM

O preço sobre lucro é um indicador fundamentalista para análise de empresas, com ele sabemos o tempo de retorno do investimento, em teoria quanto menor melhor.

Neste estudo vamos coletar os dados da DRE (Demonstrativo do Resultado do Exercício) de empresas listadas na B3, os dados serão coletados diretamente do site da CVM.

## Dependências e Configurações iniciais

In [1]:
from zipfile import ZipFile
import tempfile
import requests
from pathlib import Path
from loguru import logger

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

logger.info('BIBLIOTECAS CARREGADAS')

2026-06-10 19:41:38.932 | INFO     | __main__:<module>:12 - BIBLIOTECAS CARREGADAS


In [ ]:
DATA            = Path().cwd()/'dados'
DATA.mkdir(parents=True, exist_ok=True)
logger.info('DIRETORIO PARA DADOS CRIADO')


URL_BASE        = 'https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/'
ANO_INICIO      = 2010
ANO_FIM         = 2025
DEMONSTRATIVO   = 'DRE_con'
ARQUIVOS_ZIP    = [ f'dfp_cia_aberta_{ano}.zip' for ano in range(ANO_INICIO, ANO_FIM + 1) ]

logger.info('CONSTANTES CARREGADAS')

2026-06-10 20:14:52.041 | INFO     | __main__:<module>:3 - DIRETORIO PARA DADOS CRIADO
2026-06-10 20:14:52.052 | INFO     | __main__:<module>:14 - CONSTANTES CARREGADAS


## 1. Coleta de Arquivos na CVM

In [10]:
logger.info('Iniciando download e extração de arquivos CSVs')
logger.info(f'Total de arquivos em formato zip para download: {len(ARQUIVOS_ZIP)}')

# temp
with tempfile.TemporaryDirectory() as temp:
    temp_path = Path(temp)
    logger.debug(f'Pasta temporária criada em: {temp_path}')

    # loop para baixar e extrair 
    for arquivo in ARQUIVOS_ZIP:
        url_completa = URL_BASE + arquivo
        caminho_zip = temp_path / arquivo
        
        try:            
            logger.info(f'Baixando: {arquivo}')                             
            resposta = requests.get(url_completa, stream=True)       
            resposta.raise_for_status() 
            
            # arq em modo de escrita binária ('wb') e salva em pedaços (chunks)
            with open(caminho_zip, 'wb') as f:
                for chunk in resposta.iter_content(chunk_size=8192): 
                    if chunk: 
                        f.write(chunk)
            
            # extracao
            logger.info(f'Extraindo {arquivo}...')
            with ZipFile(caminho_zip, 'r') as zip_ref:
                zip_ref.extractall(DATA)
                
        except requests.exceptions.HTTPError as http_err:            
            logger.exception(f'Erro HTTP ao tentar baixar {arquivo}')
        except Exception as e:            
            logger.exception(f'Erro genérico ao processar {arquivo}')

logger.debug(f'Arquivos extraídos em: {DATA}')
logger.info('Processo Concluído com sucesso')

2026-06-10 19:56:34.363 | INFO     | __main__:<module>:1 - Iniciando download e extração de arquivos CSVs
2026-06-10 19:56:34.364 | INFO     | __main__:<module>:2 - Total de arquivos em formato zip para download: 16
2026-06-10 19:56:34.367 | DEBUG    | __main__:<module>:7 - Pasta temporária criada em: C:\Users\wsant\AppData\Local\Temp\tmp9o3etche
2026-06-10 19:56:34.367 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_aberta_2010.zip
2026-06-10 19:56:35.182 | INFO     | __main__:<module>:26 - Extraindo dfp_cia_aberta_2010.zip...
2026-06-10 19:56:37.850 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_aberta_2011.zip
2026-06-10 19:56:38.418 | INFO     | __main__:<module>:26 - Extraindo dfp_cia_aberta_2011.zip...
2026-06-10 19:56:40.600 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_aberta_2012.zip
2026-06-10 19:56:41.188 | INFO     | __main__:<module>:26 - Extraindo dfp_cia_aberta_2012.zip...
2026-06-10 19:56:43.296 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_

## 2. Coletando DRE

In [13]:
logger.info('Buscando arquivos CSV com DRE')

arquivos_csv     = list(DATA.glob('*.csv'))
arquivos_csv_dre = [arquivo for arquivo in arquivos_csv if DEMONSTRATIVO in arquivo.name]

if not arquivos_csv_dre:
    logger.warning('Nenhum arquivo csv com DRE encontrado.')

logger.info(f'Total de arquivos encontrados: {len(arquivos_csv_dre)}')

logger.info('Criando lista de dataframes')
lista_df = [
    pd.read_csv(arquivo, sep=';', encoding='ISO-8859-1', decimal=',', dtype=str) # dtype str :: evitar erro de tipos no concat
        for arquivo in arquivos_csv_dre
]

logger.info('Iniciar Concatenação')
df = pd.concat(lista_df, ignore_index=True)
logger.info(f'DataFrame concatenado, o dataset possui: {df.shape[0]} linhas e {df.shape[1]} colunas.')

2026-06-10 20:19:12.350 | INFO     | __main__:<module>:1 - Buscando arquivos CSV com DRE
2026-06-10 20:19:12.357 | INFO     | __main__:<module>:9 - Total de arquivos encontrados: 16
2026-06-10 20:19:12.359 | INFO     | __main__:<module>:11 - Criando lista de dataframes
2026-06-10 20:19:16.213 | INFO     | __main__:<module>:17 - Iniciar Concatenação
2026-06-10 20:19:16.239 | INFO     | __main__:<module>:19 - DataFrame concatenado, o dataset possui: 449570 linhas e 15 colunas.


In [14]:
df.head()

,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA
0,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2009-01-01,2009-12-31,3.01,Receitas da Intermediação Financeira,67608506.0000000000,S
1,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.01,Receitas da Intermediação Financeira,85143206.0000000000,S
2,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2009-01-01,2009-12-31,3.01.01,Receita de Juros,67608506.0000000000,S
3,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.01.01,Receita de Juros,85143206.0000000000,S
4,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2009-01-01,2009-12-31,3.02,Despesas da Intermediação Financeira,-39302642.0000000000,S


## 3. Tratamento dos Dados DRE

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 449570 entries, 0 to 449569
Data columns (total 15 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   CNPJ_CIA       449570 non-null  object
 1   DT_REFER       449570 non-null  object
 2   VERSAO         449570 non-null  object
 3   DENOM_CIA      449570 non-null  object
 4   CD_CVM         449570 non-null  object
 5   GRUPO_DFP      449570 non-null  object
 6   MOEDA          449570 non-null  object
 7   ESCALA_MOEDA   449570 non-null  object
 8   ORDEM_EXERC    449570 non-null  object
 9   DT_INI_EXERC   449570 non-null  object
 10  DT_FIM_EXERC   449570 non-null  object
 11  CD_CONTA       449570 non-null  object
 12  DS_CONTA       449570 non-null  object
 13  VL_CONTA       449570 non-null  object
 14  ST_CONTA_FIXA  449570 non-null  object
dtypes: object(15)
memory usage: 51.4+ MB


In [16]:
df = (df
    .assign(
        DT_REFER     = pd.to_datetime(df['DT_REFER']),
        DT_INI_EXERC = pd.to_datetime(df['DT_INI_EXERC']),
        DT_FIM_EXERC = pd.to_datetime(df['DT_FIM_EXERC']),
        VERSAO       = pd.to_numeric(df['VERSAO']),
        VL_CONTA     = pd.to_numeric(df['VL_CONTA']),
    )
)
df.dtypes

CNPJ_CIA                 object
DT_REFER         datetime64[ns]
VERSAO                    int64
DENOM_CIA                object
CD_CVM                   object
GRUPO_DFP                object
MOEDA                    object
ESCALA_MOEDA             object
ORDEM_EXERC              object
DT_INI_EXERC     datetime64[ns]
DT_FIM_EXERC     datetime64[ns]
CD_CONTA                 object
DS_CONTA                 object
VL_CONTA                float64
ST_CONTA_FIXA            object
dtype: object

In [17]:
df['ORDEM_EXERC'].unique()

array(['PENÚLTIMO', 'ÚLTIMO'], dtype=object)

In [18]:
df = df[ df['ORDEM_EXERC'] =='ÚLTIMO']

In [19]:
df['ORDEM_EXERC'].unique()

array(['ÚLTIMO'], dtype=object)

In [20]:
df.shape

(225197, 15)

## 4. Empresas 

In [22]:
df.sample()

,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA
210620,08.534.605/0001-74,2018-12-31,1,RENOVA ENERGIA S.A. - EM RECUPERAÇÃO JUDICIAL,021636,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2018-01-01,2018-12-31,3.04.06.05,Outras receitas,0.0,N


In [23]:
df['DENOM_CIA'].drop_duplicates()

1                                           BCO BRASIL S.A.
78                               BRB BANCO DE BRASILIA S.A.
113                    CENTRAIS ELET BRAS S.A. - ELETROBRAS
199                  COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB
281                                          VBC ENERGIA SA
                                ...                        
438679    ELEA DIGITAL INFRAESTRUTURA E REDES DE TELECOM...
441651                                            B100 S.A.
441803                                           REVEE S.A.
442363                        TUPI ENERGIAS RENOVÁVEIS S.A.
442509             ORANJEBTC S.A. - EDUCAÇÃO E INVESTIMENTO
Name: DENOM_CIA, Length: 769, dtype: object

In [28]:
cols = ['CD_CVM','DENOM_CIA']
empresas = df[cols].drop_duplicates()
empresas = empresas.set_index('CD_CVM')

empresas

,DENOM_CIA
CD_CVM,
001023,BCO BRASIL S.A.
014206,BRB BANCO DE BRASILIA S.A.
002437,CENTRAIS ELET BRAS S.A. - ELETROBRAS
014451,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB
014540,VBC ENERGIA SA
...,...
026948,ELEA DIGITAL INFRAESTRUTURA E REDES DE TELECOM...
027634,B100 S.A.
027650,REVEE S.A.


## 5. Coletando DRE de uma empresa

In [54]:
cia  = 'itaúsa'
mask = empresas['DENOM_CIA'].str.contains(cia, case=False, na=False)
empresas[ mask ]

,DENOM_CIA
CD_CVM,
007617,ITAÚSA S.A.


In [55]:
df[df['CD_CVM'] == '007617' ]['DENOM_CIA'].unique()

array(['ITAÚSA S.A.'], dtype=object)

In [56]:
dre = df[df['CD_CVM'] == '007617' ]

dre

,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA
20638,61.532.644/0001-15,2010-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.01,Receita de Venda de Bens e/ou Serviços,5.240000e+06,S
20640,61.532.644/0001-15,2010-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.02,Custo dos Bens e/ou Serviços Vendidos,-3.624000e+06,S
20642,61.532.644/0001-15,2010-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.03,Resultado Bruto,1.616000e+06,S
20644,61.532.644/0001-15,2010-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.04,Despesas/Receitas Operacionais,-1.583000e+07,S
20646,61.532.644/0001-15,2010-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.04.01,Despesas com Vendas,0.000000e+00,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444671,61.532.644/0001-15,2025-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2025-01-01,2025-12-31,3.99.01.01,ON,1.477140e+00,N
444673,61.532.644/0001-15,2025-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2025-01-01,2025-12-31,3.99.01.02,PN,1.477140e+00,N
444675,61.532.644/0001-15,2025-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2025-01-01,2025-12-31,3.99.02,Lucro Diluído por Ação,0.000000e+00,N
444677,61.532.644/0001-15,2025-12-31,1,ITAÚSA S.A.,007617,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2025-01-01,2025-12-31,3.99.02.01,ON,1.477140e+00,N


## 6. Contas do DRE

In [ ]:
contas_itausa = df \
    .loc[df['CD_CVM'] == '007617', ['CD_CONTA', 'DS_CONTA']] \
    .drop_duplicates() \
    .set_index("CD_CONTA")


contas_itausa

,DS_CONTA
CD_CONTA,
3.01,Receita de Venda de Bens e/ou Serviços
3.02,Custo dos Bens e/ou Serviços Vendidos
3.03,Resultado Bruto
3.04,Despesas/Receitas Operacionais
3.04.01,Despesas com Vendas
3.04.02,Despesas Gerais e Administrativas
3.04.03,Perdas pela Não Recuperabilidade de Ativos
3.04.04,Outras Receitas Operacionais
3.04.05,Outras Despesas Operacionais


## 7. Coletando Lucro por Ação

In [79]:
# dre[dre['CD_CONTA'] == '3.99.01.02']
lpa = df \
        .loc[(df['CD_CVM'] == '007617') & (df['CD_CONTA'] == '3.99.01.02'), ['DT_REFER', 'VL_CONTA']] \
        .set_index('DT_REFER')
lpa

,VL_CONTA
DT_REFER,
2010-12-31,1.01000
2011-12-31,1.10000
2012-12-31,0.94000
2013-12-31,1.05000
2014-12-31,1.30000
2015-12-31,1.31000
2016-12-31,1.11000
2017-12-31,1.13000
2018-12-31,1.13000


## 8. Coleta de Cotações

In [86]:
precos = yf.download('ITSA4.SA', start='2011-01-01', auto_adjust=False)

[*********************100%***********************]  1 of 1 completed


In [91]:
precos[['Adj Close', 'Close']]

Price,Adj Close,Close
Ticker,ITSA4.SA,ITSA4.SA
Date,,
2011-01-03,2.115089,5.559474
2011-01-04,2.154424,5.631019
2011-01-05,2.181797,5.702564
2011-01-06,2.135102,5.580516
2011-01-07,2.072304,5.416384
...,...,...
2026-06-03,12.600000,12.600000
2026-06-05,12.540000,12.540000
